# 05 - Robustness (NC), format-swap (NC), explainability (SM), campus test (NG)

Run only the part that is yours. Each part loads a trained model from `reports/run_X/best.pt` (or `models/final.pt`).

* **Part A (NC):** how much does accuracy fall under compression, resizing, blur, noise, darkness?
* **Part B (NC):** format-swap test: re-save images in the other format. If accuracy collapses, the model was reading file types.
* **Part C (SM):** Grad-CAM heat maps and the *faithfulness* test (does hiding the highlighted part really change the answer more than hiding a random part?).
* **Part D (NG, ONCE):** the locked UCU Mukono campus set, in three versions (original, WhatsApp photo, WhatsApp document).

In [ ]:
# CELL 1 - run this first. It downloads the project code into this Colab session.
import os, subprocess, sys
REPO_URL = "https://github.com/andrinawatsemba/DEEPFAKE_CV_INVESTIGATION.git"
CODE = "/content/repo"
if not os.path.isdir(CODE + "/src"):
    for branch in ("main", "gw/lean-pipeline"):          # main first; the branch is only a fallback
        subprocess.run(["rm", "-rf", CODE])
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, CODE])
        if r.returncode == 0 and os.path.exists(CODE + "/src/colab_setup.py"):
            print("Got the code from branch:", branch)
            break
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"])
sys.path.insert(0, CODE + "/src")
print("Code ready.")


In [ ]:
# CELL 2 - connect, load a model and the test images
import os, json, torch, pandas as pd, numpy as np
from colab_setup import setup
from model import load_checkpoint
ROOT, REPO = setup()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN = "run_E"                     # which run's model to examine
model, _ = load_checkpoint(f"{ROOT}/reports/{RUN}/best.pt", DEVICE)
THR = json.load(open(f"{ROOT}/reports/{RUN}/metrics.json"))["_run"]["threshold"]
SPLIT = pd.read_csv(f"{ROOT}/audit_outputs/split.csv")
TEST = SPLIT[SPLIT.role.isin(["test_seen", "test_unseen"])]
SAMPLE = pd.concat([g.sample(min(len(g), 75), random_state=0) for _, g in TEST.groupby("label")])   # 150 images keeps it fast
print(len(SAMPLE), "test images; threshold", THR)

## Part A (NC): robustness under damage

In [ ]:
# CELL 3 - Part A. Takes a few minutes.
import stress, matplotlib.pyplot as plt
T = stress.stress_table(model, SAMPLE, ROOT, threshold=THR, device=DEVICE)
display(T.round(3))
os.makedirs(f"{ROOT}/reports/robustness", exist_ok=True)
T.to_csv(f"{ROOT}/reports/robustness/{RUN}_stress.csv", index=False)
ax = T.set_index("condition").macro_f1.plot.barh(figsize=(7, 5), color="#A64B14")
ax.set_xlabel("macro-F1"); ax.set_title(f"{RUN}: accuracy under damage"); plt.tight_layout()
plt.savefig(f"{ROOT}/reports/robustness/{RUN}_stress.png", dpi=130); plt.show()

## Part B (NC): the format-swap test

In [ ]:
# CELL 4 - Part B. JPEG images saved losslessly as PNG, and everything saved as high-quality JPEG.
FS = stress.stress_table(model, SAMPLE, ROOT, conditions={"original": stress.CONDITIONS["original"], **stress.FORMAT_SWAP},
                         threshold=THR, device=DEVICE)
display(FS.round(3))
FS.to_csv(f"{ROOT}/reports/robustness/{RUN}_format_swap.csv", index=False)
print("If macro_f1 drops a lot for as_png or as_jpeg95, the model was using the file format.")

## Part C (SM): Grad-CAM and the faithfulness test

In [ ]:
# CELL 5 - Part C. Heat maps for 6 images.
from explain import GradCAM, overlay
from data import load_image, denormalise
import matplotlib.pyplot as plt
rows = SAMPLE.sample(6, random_state=1)
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
cam_fn = GradCAM(model, model.gradcam_layer())
for j, r in enumerate(rows.itertuples()):
    x, _ = load_image(f"{ROOT}/{r.path}", mode="center")
    cam, p = cam_fn(x.to(DEVICE))
    ax[0, j].imshow(denormalise(x[0])); ax[0, j].set_title(f"{r.label}", fontsize=9); ax[0, j].axis("off")
    ax[1, j].imshow(overlay(denormalise(x[0]), cam)); ax[1, j].set_title(f"p(AI)={p:.2f}", fontsize=9); ax[1, j].axis("off")
cam_fn.close(); plt.tight_layout(); plt.show()

In [ ]:
# CELL 6 - THE FAITHFULNESS TEST (100 images: 50 real + 50 AI). A few minutes.
from explain import faithfulness_study
FSAMPLE = pd.concat([g.sample(min(len(g), 50), random_state=2) for _, g in TEST.groupby("label")])
per_image, summary = faithfulness_study(model, FSAMPLE, ROOT, DEVICE, frac=0.2, n_random=5)
print(json.dumps(summary, indent=1))
os.makedirs(f"{ROOT}/reports/explainability", exist_ok=True)
per_image.to_csv(f"{ROOT}/reports/explainability/{RUN}_faithfulness.csv", index=False)
json.dump(summary, open(f"{ROOT}/reports/explainability/{RUN}_faithfulness.json", "w"), indent=2)
plt.figure(figsize=(5, 4))
plt.bar(["hide Grad-CAM region", "hide random region"], [summary["mean_drop_cam"], summary["mean_drop_random"]], color=["#A64B14", "#56636E"])
plt.ylabel("mean drop in confidence"); plt.title("Does the heat map point at what matters?"); plt.tight_layout()
plt.savefig(f"{ROOT}/reports/explainability/{RUN}_faithfulness.png", dpi=130); plt.show()
print(summary.get("verdict"))

## Part D (NG): the campus test, ONCE
This opens the locked set. **Only run it with the final model, after the model is chosen.** Do not change the model and run it again to improve the score.

In [ ]:
# CELL 7 - Part D. Set the flag to True only when you are sure.
I_UNDERSTAND_THIS_RUNS_ONCE = False
if I_UNDERSTAND_THIS_RUNS_ONCE:
    import campus_tools as C
    from evaluate import evaluate_roles
    CAMP = C.test_table(ROOT)
    m, p = evaluate_roles(model, CAMP, ROOT, sorted(CAMP.role.unique()), THR, DEVICE)
    os.makedirs(f"{ROOT}/reports/campus_test", exist_ok=True)
    json.dump(m, open(f"{ROOT}/reports/campus_test/{RUN}_metrics.json", "w"), indent=2)
    p.to_csv(f"{ROOT}/reports/campus_test/{RUN}_predictions.csv", index=False)
    display(pd.DataFrame({k: {x: v[x] for x in ("n", "accuracy", "macro_f1", "recall_real", "recall_ai")} for k, v in m.items()}).T.round(3))
else:
    print("Not run. Set I_UNDERSTAND_THIS_RUNS_ONCE = True when the final model is frozen.")